In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Qiskit Runtime lokal na testing mode

<span id="test-locally"></span>


<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na requirements.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago pa.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
```
</details>
Gamitin ang lokal na testing mode (available sa `qiskit-ibm-runtime` v0.22.0 o mas bago) para subukan ang mga programa bago i-fine-tune at ipadala sa tunay na quantum hardware. Pagkatapos gamitin ang lokal na testing mode para i-verify ang iyong programa, ang kailangan mo lang baguhin ay ang pangalan ng backend para patakbuhin ito sa isang QPU.

Para gamitin ang lokal na testing mode, tukuyin ang isa sa mga fake backend mula sa ``qiskit_ibm_runtime.fake_provider`` o tukuyin ang isang Qiskit Aer backend kapag nag-instantiate ng Qiskit Runtime primitive o session.

- **Mga fake backend**: Ang mga [fake backend](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider) sa ``qiskit_ibm_runtime.fake_provider`` ay ginagaya ang mga gawi ng IBM&reg; QPU sa pamamagitan ng paggamit ng mga QPU snapshot. Ang mga QPU snapshot ay naglalaman ng mahahalagang impormasyon tungkol sa QPU, tulad ng coupling map, basis gates, at mga katangian ng qubit, na kapaki-pakinabang sa pagsubok ng transpiler at pagsasagawa ng maingay na simulation ng QPU. Ang noise model mula sa snapshot ay awtomatikong inilalapat sa panahon ng simulation.
- **Aer simulator**: Ang mga simulator mula sa [Qiskit Aer](/guides/simulate-with-qiskit-aer) ay nagbibigay ng mas mataas na performance na simulation na kayang hawakan ang mas malalaking Circuit at [custom na noise model](/guides/build-noise-models). Makikita ang listahan ng mga opsyon sa simulation method kapag ginamit mo ang `AerSimulator` sa lokal na testing mode. Tingnan ang [halimbawa ng Clifford simulation mode](#clifford-sim), na nagpapakita kung paano mahusay na mag-simulate ng mga Clifford Circuit na may malaking bilang ng qubit.

    <details>
    <summary>**Listahan ng mga simulation method na available mula sa Qiskit Aer**</summary>

    Tingnan ang dokumentasyon ng [`AerSimulator`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator) para sa karagdagang impormasyon.

    * ``"automatic"``: Default na simulation method. Awtomatikong pinipili ang simulation method batay sa Circuit at noise model.

    * ``"statevector"``: Isang masinsinang statevector simulation na kayang mag-sample ng mga resulta ng pagsukat mula sa mga *ideal* na Circuit na may lahat ng pagsukat sa dulo ng Circuit. Para sa maingay na simulation, bawat shot ay nag-sa-sample ng random na maingay na Circuit mula sa noise model.

    * ``"density_matrix"``: Isang density matrix simulation na kayang mag-sample ng mga resulta ng pagsukat mula sa mga *maingay* na Circuit na may lahat ng pagsukat sa dulo ng Circuit.

    * ``"stabilizer"``: Isang mahusay na Clifford stabilizer state simulator na kayang mag-simulate ng maingay na Clifford Circuit kung lahat ng error sa noise model ay Clifford error din.

    * ``"extended_stabilizer"``: Isang approximate simulator para sa mga Clifford + T Circuit batay sa pag-decompose ng state sa isang ranked-stabilizer state. Lumalaki ang bilang ng termino habang dumarami ang mga non-Clifford (T) Gate.

    * ``"matrix_product_state"``: Isang tensor-network statevector simulator na gumagamit ng Matrix Product State (MPS) na representasyon para sa state. Magagawa ito nang may o walang pag-truncate ng MPS bond dimensions, depende sa mga opsyon ng simulator. Ang default na gawi ay walang truncation.

    * ``"unitary"``: Isang masinsinang unitary matrix simulation ng isang ideal na Circuit. Sine-simulate nito ang unitary matrix ng Circuit mismo, sa halip na ang ebolusyon ng isang paunang quantum state. Ang method na ito ay kayang mag-simulate lamang ng mga Gate; hindi ito sumusuporta sa pagsukat, reset, o noise.

    * ``"superop"``: Isang masinsinang superoperator matrix simulation ng isang ideal o maingay na Circuit. Sine-simulate nito ang superoperator matrix ng Circuit mismo, sa halip na ang ebolusyon ng isang paunang quantum state. Ang method na ito ay kayang mag-simulate ng ideal at maingay na Gate at reset, ngunit hindi ito sumusuporta sa mga pagsukat.

    * ``"tensor_network"``: Isang tensor-network-based simulation na sumusuporta sa parehong statevector at density matrix. Sa kasalukuyan, available lamang ito para sa GPU at pinabilis gamit ang cuQuantum `cuTensorNet` API.
    </details>

> **Note:** - Maaari mong tukuyin ang lahat ng opsyon ng Qiskit Runtime sa lokal na testing mode. Gayunpaman, lahat ng opsyon maliban sa shots ay binabalewala kapag pinatakbo sa lokal na simulator.
> - Inirerekomenda na i-install mo ang Qiskit Aer bago gamitin ang mga fake backend o Aer simulator sa pamamagitan ng pagpapatakbo ng `pip install qiskit-aer`. Ang mga fake backend ay gumagamit ng mga Aer simulator sa likod ng eksena kung available, para mapakinabangan ang kanilang performance.


## Halimbawa ng mga fake backend

In [1]:
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using FakeManilaV2
fake_manila = FakeManilaV2()
pm = generate_preset_pass_manager(backend=fake_manila, optimization_level=1)
isa_qc = pm.run(qc)

# You can use a fixed seed to get fixed results.
options = {"simulator": {"seed_simulator": 42}}
sampler = Sampler(mode=fake_manila, options=options)

result = sampler.run([isa_qc]).result()

## Mga halimbawa ng AerSimulator {#aerSimulator-examples}

Halimbawa na may mga session, walang noise:

In [2]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using AerSimulator.
# Session syntax is supported but ignored because local mode doesn't support sessions.
aer_sim = AerSimulator()
pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
with Session(backend=aer_sim) as session:
    sampler = Sampler(mode=session)
    result = sampler.run([isa_qc]).result()

Para mag-simulate na may noise, tukuyin ang isang QPU (quantum hardware) at isumite ito sa Aer. Nagtatayo ang Aer ng noise model batay sa calibration data mula sa QPU na iyon, at nag-i-instantiate ng Aer backend na may modelong iyon. Kung gusto mo, maaari kang [gumawa ng noise model](/guides/build-noise-models).

> **Caution:** Ang isang QPU ay maaaring maapektuhan ng iba't ibang uri ng noise. Ang Qiskit Aer noise model na ginagamit dito ay nag-si-simulate lamang ng ilan sa mga ito at samakatuwid ay malamang na hindi kasing grabe ng noise sa isang tunay na QPU.
> 
> Para sa mga detalye kung aling mga error ang kasama kapag nagsi-initialize ng noise model mula sa isang QPU, tingnan ang Aer [`NoiseModel`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.NoiseModel.html#qiskit_aer.noise.NoiseModel.from_backend) API reference.

Halimbawa na may noise:

In [3]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

service = QiskitRuntimeService()

# Specify a QPU to use for the noise model
real_backend = service.backend("ibm_fez")
aer = AerSimulator.from_backend(real_backend)

# Run the sampler job locally using AerSimulator.
pm = generate_preset_pass_manager(backend=aer, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer)
result = sampler.run([isa_qc]).result()

<span id="clifford-sim"></span>
### Clifford simulation
Dahil ang mga Clifford Circuit ay maaaring mahusay na ma-simulate na may mga nabeberipikang resulta, ang Clifford simulation ay isang napaka-kapaki-pakinabang na kasangkapan. Para sa isang malalim na halimbawa, tingnan ang [Mahusay na simulation ng mga stabilizer Circuit gamit ang mga Qiskit Aer primitive](/guides/simulate-stabilizer-circuits).

Halimbawa:

In [4]:
import numpy as np
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import SamplerV2 as Sampler

n_qubits = 500  # <---- note this uses 500 qubits!
circuit = efficient_su2(n_qubits)
circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Tell Aer to use the stabilizer (Clifford) simulation method
aer_sim = AerSimulator(method="stabilizer")

pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer_sim)
result = sampler.run([isa_qc]).result()

## Mga susunod na hakbang
> **Tip:** - Suriin ang mga detalyadong [halimbawa ng primitive.](/guides/primitives-examples)
>     - Basahin ang [Mag-migrate sa V2 primitives](/guides/v2-primitives).
>     - Mag-practice gamit ang mga primitive sa pamamagitan ng pagtatapos ng [aralin sa Cost function](/learning/courses/variational-algorithm-design/cost-functions) sa IBM Quantum Learning.
>     - Alamin kung paano mag-transpile nang lokal sa seksyong [Transpile](/guides/transpile).
>     - Subukan ang tutorial na [Ihambing ang mga setting ng transpiler](/guides/circuit-transpilation-settings#compare-transpiler-settings).